In [1]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk("/kaggle/input/playground-series-s3e18"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/playground-series-s3e18/train.csv
/kaggle/input/playground-series-s3e18/description.md
/kaggle/input/playground-series-s3e18/sample_submission.csv.zip
/kaggle/input/playground-series-s3e18/test.csv.zip
/kaggle/input/playground-series-s3e18/train.csv.zip
/kaggle/input/playground-series-s3e18/test.csv
/kaggle/input/playground-series-s3e18/sample_submission.csv


In [2]:
submission = pd.read_csv("/kaggle/input/playground-series-s3e18/sample_submission.csv")
train = pd.read_csv("/kaggle/input/playground-series-s3e18/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s3e18/test.csv")

print("train:", train.shape, "test:", test.shape, "sample_sub:", submission.shape)
print("submission columns:", submission.columns.tolist())



train: (13354, 38) test: (1484, 32) sample_sub: (1484, 3)
submission columns: ['id', 'EC1', 'EC2']


In [3]:
target_cols = ["EC1", "EC2"]
all_target_cols = [c for c in train.columns if c.startswith("EC") and c[2:].isdigit()]

feature_cols = [c for c in train.columns if c not in (["id"] + all_target_cols)]

common_feature_cols = [c for c in feature_cols if c in test.columns]
missing_in_test = [c for c in feature_cols if c not in test.columns]
extra_in_test = [
    c for c in test.columns if (c not in common_feature_cols and c != "id")
]

print("n_features_train_defined:", len(feature_cols))
print("n_features_common_used:", len(common_feature_cols))
if missing_in_test:
    print(
        "Dropped (missing in test):",
        missing_in_test[:10],
        "..." if len(missing_in_test) > 10 else "",
    )
if extra_in_test:
    print(
        "Unused extra test cols (not in train features):",
        extra_in_test[:10],
        "..." if len(extra_in_test) > 10 else "",
    )

X_train = train[common_feature_cols].copy()
X_test = test[common_feature_cols].copy()

y_train_ec1 = train["EC1"].astype(int).values
y_train_ec2 = train["EC2"].astype(int).values

print("X_train:", X_train.shape, "X_test:", X_test.shape)



n_features_train_defined: 31
n_features_common_used: 31
X_train: (13354, 31) X_test: (1484, 31)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

base_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        (
            "clf",
            LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                C=1.0,
                random_state=42,
            ),
        ),
    ]
)



In [5]:
# Change (score-related): use Stratified CV to prevent any preprocessing leakage and to average
# per-fold test probabilities. This preserves the same core logic (same pipeline/model), but
# usually improves AUC vs fitting once on all data with a single deterministic scaling.
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone


def cv_predict_proba_and_test_average(X, y, X_test, pipe, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(X.shape[0], dtype=float)
    test_pred = np.zeros(X_test.shape[0], dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        m = clone(pipe)
        m.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / n_splits

        if fold == 1:
            print(
                "Fold 1 sanity - test pred range:",
                float(test_pred.min()),
                float(test_pred.max()),
            )

    return oof, test_pred


oof_ec1, pred_ec1 = cv_predict_proba_and_test_average(
    X_train, y_train_ec1, X_test, base_pipe, n_splits=5, seed=42
)
oof_ec2, pred_ec2 = cv_predict_proba_and_test_average(
    X_train, y_train_ec2, X_test, base_pipe, n_splits=5, seed=42
)

print("pred_ec1 range:", float(pred_ec1.min()), float(pred_ec1.max()))
print("pred_ec2 range:", float(pred_ec2.min()), float(pred_ec2.max()))



Fold 1 sanity - test pred range: 0.0263866133321727 0.18976441400804078
Fold 1 sanity - test pred range: 0.07870647570763437 0.19197342713182344
pred_ec1 range: 0.1417323803103416 0.9408296318059419
pred_ec2 range: 0.4773751043442226 0.9383314642133348


In [6]:
sub = submission.copy()

if "id" in sub.columns and "id" in test.columns:
    if not sub["id"].equals(test["id"]):
        sub = sub.set_index("id").reindex(test["id"].values).reset_index()

sub["EC1"] = pred_ec1
sub["EC2"] = pred_ec2

sub = sub[["id", "EC1", "EC2"]]
assert sub.shape[0] == test.shape[0], "Submission row count mismatch vs test."
assert list(sub.columns) == ["id", "EC1", "EC2"], "Submission columns mismatch."

out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)
print("Wrote:", out_path)
print(sub.head())

Wrote: /kaggle/working/submission.csv
      id       EC1       EC2
0   1377  0.761484  0.743335
1   5485  0.683538  0.828580
2   8753  0.584525  0.834758
3  10746  0.713550  0.739614
4   7554  0.488481  0.845051
